# Kaggle Playground Series S6E5 — Predict F1 Pit Stops
### Author: Ruide Yin

## Stage 3-5: Full Pipeline — Tuning & Ensemble

Continues from Stage 2 - feature engineering and model training. 

### 3.0 Setup, Load Data & Load Models

In [1]:
import pandas as pd, numpy as np, os, time, warnings, gc, joblib, pickle
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
warnings.filterwarnings('ignore')

SEED = 12138
N_FOLDS = 5
TARGET = 'PitNextLap'

X = np.load('./outputs/X.npy')
y = np.load('./outputs/y.npy')
X_test = np.load('./outputs/X_test.npy')
X_scaled = np.load('./outputs/X_scaled.npy')
X_test_scaled = np.load('./outputs/X_test_scaled.npy')
X_filled = np.load('./outputs/X_filled.npy')
X_test_filled = np.load('./outputs/X_test_filled.npy')
test_ids = np.load('./outputs/test_ids.npy')
FEATURES = pickle.load(open('./outputs/features_list.pkl', 'rb'))
results = pickle.load(open('./outputs/baseline_results.pkl', 'rb'))
scale_pos = pickle.load(open('./outputs/scale_pos.pkl', 'rb'))
scaler = joblib.load('./outputs/scaler.pkl')
print(f"Loaded: X {X.shape}, y {y.shape}, {len(results)} baseline models")
print(f"scale_pos_weight = {scale_pos:.2f}")

d:\Users\cnyrd\SWAGGYYIN\NYU\Data Science Projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded: X (540511, 151), y (540511,), 6 baseline models
scale_pos_weight = 3.77


In [2]:
# ── New artifacts from updated 02_train ─────────────────────────────────────
X_cat_df = pd.read_parquet('./outputs/X_cat_df.parquet')
X_test_cat_df = pd.read_parquet('./outputs/X_test_cat_df.parquet')
CAT_COLS = pickle.load(open('./outputs/cat_cols.pkl', 'rb'))
CAT_INDICES = pickle.load(open('./outputs/cat_indices.pkl', 'rb'))
ALL_FEATURES = pickle.load(open('./outputs/all_features.pkl', 'rb'))

groups = np.load('./outputs/groups.npy', allow_pickle=True)
print(f"Loaded {len(np.unique(groups))} groups for StratifiedGroupKFold CV")

spw = pickle.load(open('./outputs/scale_pos_winners.pkl', 'rb'))
LGB_SPW = spw['LGB_BEST_SCALE_POS']
XGB_SPW = spw['XGB_BEST_SCALE_POS']
print(f"LGB tuning will use scale_pos_weight={LGB_SPW:.2f}")
print(f"XGB tuning will use scale_pos_weight={XGB_SPW:.2f}")

# ── Helper: switchable splitter ─────────────────────────────────────────────
def get_splits(X_arr, y_arr, groups=None, n_folds=N_FOLDS, seed=SEED):
    """StratifiedGroupKFold if groups provided, else StratifiedKFold."""
    if groups is not None:
        sgkf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
        return list(sgkf.split(X_arr, y_arr, groups=groups))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    return list(skf.split(X_arr, y_arr))

# ── Probe LightGBM GPU availability (CUDA → OpenCL → CPU fallback) ──────────
LGB_DEVICE = 'cpu'
try:
    _params = {'device': 'cuda', 'objective': 'binary', 'metric': 'auc', 'verbosity': -1}
    _dtr = lgb.Dataset(X[:1000], y[:1000])
    _ = lgb.train(_params, _dtr, num_boost_round=5)
    LGB_DEVICE = 'cuda'
    print("LGB CUDA works")
except Exception as e:
    print(f"LGB CUDA failed: {str(e)[:80]}")
    try:
        _params['device'] = 'gpu'
        _dtr = lgb.Dataset(X[:1000], y[:1000])
        _ = lgb.train(_params, _dtr, num_boost_round=5)
        LGB_DEVICE = 'gpu'
        print("LGB OpenCL GPU works")
    except Exception as e2:
        print(f"LGB GPU failed: {str(e2)[:80]}; falling back to CPU")
        LGB_DEVICE = 'cpu'
print(f">>> LGB_DEVICE = {LGB_DEVICE}")

# ── Build group-CV splits once, reuse everywhere ────────────────────────────
splits_5fold = get_splits(X, y, groups=groups, n_folds=N_FOLDS, seed=SEED)
print(f"Built {N_FOLDS} group-aware splits, sizes: "
      f"{[len(v) for _, v in splits_5fold]}")

# ── CatBoost trainer (DataFrame + cat_features + NaN preserved) ─────────────
def train_catboost_model(params, X_tr_df, y_tr, X_te_df, name, cat_indices,
                         splits=None, n_folds=N_FOLDS):
    t0 = time.time()
    oof = np.zeros(len(y_tr))
    test_preds = np.zeros(len(X_te_df))
    if splits is None:
        splits = get_splits(X_tr_df, y_tr, groups=groups, n_folds=n_folds)
    for fold, (trn_idx, val_idx) in enumerate(splits):
        clf = CatBoostClassifier(**params)
        clf.fit(
            X_tr_df.iloc[trn_idx], y_tr[trn_idx],
            eval_set=(X_tr_df.iloc[val_idx], y_tr[val_idx]),
            cat_features=cat_indices,
            early_stopping_rounds=200, verbose=0,
        )
        oof[val_idx] = clf.predict_proba(X_tr_df.iloc[val_idx])[:, 1]
        test_preds += clf.predict_proba(X_te_df)[:, 1] / len(splits)
    auc = roc_auc_score(y_tr, oof)
    elapsed = time.time() - t0
    print(f"[{name}] OOF AUC = {auc:.6f} | Time = {elapsed:.1f}s")
    return oof, test_preds, auc, elapsed

print("Helpers ready.")

Loaded 106 groups for StratifiedGroupKFold CV
LGB tuning will use scale_pos_weight=1.00
XGB tuning will use scale_pos_weight=1.00
LGB CUDA failed: CUDA Tree Learner was not enabled in this build.
Please recompile with CMake opt
LGB OpenCL GPU works
>>> LGB_DEVICE = gpu
Built 5 group-aware splits, sizes: [108103, 108102, 108102, 108102, 108102]
Helpers ready.


## Part 3: Hyperparameter Tuning (Optuna)

Inner CV uses StratifiedKFold 3-fold for speed. After tuning, final models retrained with 5-fold.

### 3.1 LightGBM Tuning (20 trials)

In [3]:
def lgb_objective(trial):
    params = {
        'objective': 'binary', 'metric': 'auc', 'verbosity': -1,
        'n_jobs': -1, 'random_state': SEED,
        'device': LGB_DEVICE,
        'scale_pos_weight': LGB_SPW,
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 31, 200),
        'max_depth':         trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'subsample_freq':    1,
    }
    inner_splits = get_splits(X, y, groups=groups, n_folds=3, seed=SEED)
    aucs = []
    for trn_idx, val_idx in inner_splits:
        dtrain = lgb.Dataset(X[trn_idx], y[trn_idx])
        dval   = lgb.Dataset(X[val_idx], y[val_idx], reference=dtrain)
        model = lgb.train(params, dtrain, num_boost_round=2000,
                          valid_sets=[dval],
                          callbacks=[lgb.early_stopping(50, verbose=False),
                                     lgb.log_evaluation(0)])
        preds = model.predict(X[val_idx])
        aucs.append(roc_auc_score(y[val_idx], preds))
    return float(np.mean(aucs))

t0 = time.time()
study_lgb = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
study_lgb.optimize(lgb_objective, n_trials=20, show_progress_bar=True)
print(f"\nLGB best trial AUC: {study_lgb.best_value:.6f} ({time.time()-t0:.0f}s)")
print(f"Best params: {study_lgb.best_params}")

Best trial: 18. Best value: 0.962752: 100%|██████████| 20/20 [1:31:47<00:00, 275.38s/it]


LGB best trial AUC: 0.962752 (5508s)
Best params: {'learning_rate': 0.03362108363186173, 'num_leaves': 156, 'max_depth': 12, 'min_child_samples': 150, 'reg_alpha': 0.8214905084721366, 'reg_lambda': 0.01853117977338995, 'colsample_bytree': 0.6441396494651419, 'subsample': 0.9350905203863216}


In [4]:
def train_lgb_with_splits(params, splits, name):
    t0 = time.time()
    oof = np.zeros(len(y))
    test_preds = np.zeros(len(X_test))
    for trn_idx, val_idx in splits:
        dtrain = lgb.Dataset(X[trn_idx], y[trn_idx])
        dval   = lgb.Dataset(X[val_idx], y[val_idx], reference=dtrain)
        model = lgb.train(params, dtrain, num_boost_round=3000,
                          valid_sets=[dval],
                          callbacks=[lgb.early_stopping(100, verbose=False),
                                     lgb.log_evaluation(0)])
        oof[val_idx] = model.predict(X[val_idx])
        test_preds += model.predict(X_test) / len(splits)
    auc = roc_auc_score(y, oof)
    elapsed = time.time() - t0
    print(f"[{name}] OOF AUC = {auc:.6f} | Time = {elapsed:.1f}s")
    return oof, test_preds, auc, elapsed

best_lgb_params = {
    'objective': 'binary', 'metric': 'auc', 'verbosity': -1,
    'n_jobs': -1, 'random_state': SEED,
    'device': LGB_DEVICE,
    'scale_pos_weight': LGB_SPW,
    **study_lgb.best_params,
    'subsample_freq': 1,
}
oof_lgb_t, tp_lgb_t, auc_lgb_t, t_lgb_t = train_lgb_with_splits(
    best_lgb_params, splits_5fold, 'LGB_tuned'
)
results['LGB_tuned'] = (oof_lgb_t, tp_lgb_t, auc_lgb_t, t_lgb_t)

[LGB_tuned] OOF AUC = 0.963585 | Time = 928.9s


### 3.2 XGBoost Tuning (80 trials)

In [5]:
def xgb_objective(trial):
    params = {
        'objective': 'binary:logistic', 'eval_metric': 'auc',
        'tree_method': 'hist', 'device': 'cuda', 'seed': SEED,
        'scale_pos_weight': XGB_SPW,
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':         trial.suggest_int('max_depth', 4, 12),
        'min_child_weight':  trial.suggest_int('min_child_weight', 5, 200),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'gamma':             trial.suggest_float('gamma', 0.0, 5.0),
    }
    inner_splits = get_splits(X, y, groups=groups, n_folds=3, seed=SEED)
    aucs = []
    for trn_idx, val_idx in inner_splits:
        dtrain = xgb.DMatrix(X[trn_idx], y[trn_idx], feature_names=FEATURES)
        dval   = xgb.DMatrix(X[val_idx], y[val_idx], feature_names=FEATURES)
        model = xgb.train(params, dtrain, num_boost_round=2000,
                          evals=[(dval, 'val')],
                          early_stopping_rounds=50, verbose_eval=0)
        preds = model.predict(dval)
        aucs.append(roc_auc_score(y[val_idx], preds))
    return float(np.mean(aucs))

t0 = time.time()
study_xgb = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
study_xgb.optimize(xgb_objective, n_trials=80, show_progress_bar=True)
print(f"\nXGB best trial AUC: {study_xgb.best_value:.6f} ({time.time()-t0:.0f}s)")
print(f"Best params: {study_xgb.best_params}")

Best trial: 78. Best value: 0.962937: 100%|██████████| 80/80 [2:14:47<00:00, 101.09s/it]  


XGB best trial AUC: 0.962937 (8088s)
Best params: {'learning_rate': 0.015326594750072243, 'max_depth': 12, 'min_child_weight': 5, 'reg_alpha': 0.5756409511655697, 'reg_lambda': 0.04688329907094969, 'colsample_bytree': 0.8887435770973438, 'subsample': 0.7499571056426504, 'gamma': 1.3639845894391098}


In [6]:
def train_xgb_with_splits(params, splits, name):
    t0 = time.time()
    oof = np.zeros(len(y))
    test_preds = np.zeros(len(X_test))
    for trn_idx, val_idx in splits:
        dtrain = xgb.DMatrix(X[trn_idx], y[trn_idx], feature_names=FEATURES)
        dval   = xgb.DMatrix(X[val_idx], y[val_idx], feature_names=FEATURES)
        model = xgb.train(params, dtrain, num_boost_round=3000,
                          evals=[(dval, 'val')],
                          early_stopping_rounds=100, verbose_eval=0)
        oof[val_idx] = model.predict(dval)
        dtest = xgb.DMatrix(X_test, feature_names=FEATURES)
        test_preds += model.predict(dtest) / len(splits)
    auc = roc_auc_score(y, oof)
    elapsed = time.time() - t0
    print(f"[{name}] OOF AUC = {auc:.6f} | Time = {elapsed:.1f}s")
    return oof, test_preds, auc, elapsed

best_xgb_params = {
    'objective': 'binary:logistic', 'eval_metric': 'auc',
    'tree_method': 'hist', 'device': 'cuda', 'seed': SEED,
    'scale_pos_weight': XGB_SPW,
    **study_xgb.best_params,
}
oof_xgb_t, tp_xgb_t, auc_xgb_t, t_xgb_t = train_xgb_with_splits(
    best_xgb_params, splits_5fold, 'XGB_tuned'
)
results['XGB_tuned'] = (oof_xgb_t, tp_xgb_t, auc_xgb_t, t_xgb_t)

[XGB_tuned] OOF AUC = 0.963812 | Time = 460.5s


### 3.3 CatBoost tuning

In [10]:
import gc

def cat_objective(trial):
    params = {
        'iterations': 2000,
        'learning_rate':       trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
        'depth':               trial.suggest_int('depth', 6, 10),
        'l2_leaf_reg':         trial.suggest_float('l2_leaf_reg', 3.0, 15.0, log=True),
        'random_strength':     trial.suggest_float('random_strength', 0.3, 1.5),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.2, 1.0),
        'bootstrap_type': 'Bayesian',
        'loss_function': 'Logloss',
        'eval_metric': 'Logloss',
        'auto_class_weights': 'Balanced',
        'task_type': 'GPU',
        'gpu_ram_part': 0.85,
        'random_seed': SEED,
        'allow_writing_files': False,
        'verbose': 0,
    }
    inner_splits = get_splits(X_cat_df, y, groups=groups, n_folds=2, seed=SEED)
    aucs = []
    for trn_idx, val_idx in inner_splits:
        clf = CatBoostClassifier(**params)
        clf.fit(
            X_cat_df.iloc[trn_idx], y[trn_idx],
            eval_set=(X_cat_df.iloc[val_idx], y[val_idx]),
            cat_features=CAT_INDICES,
            early_stopping_rounds=30, verbose=0,
        )
        preds = clf.predict_proba(X_cat_df.iloc[val_idx])[:, 1]
        aucs.append(roc_auc_score(y[val_idx], preds))
        del clf
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return float(np.mean(aucs))

t0 = time.time()
study_cat = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
study_cat.optimize(cat_objective, n_trials=40, timeout=10800, show_progress_bar=True)
print(f"\nCatBoost best trial AUC: {study_cat.best_value:.6f} ({time.time()-t0:.0f}s)")
print(f"Best params: {study_cat.best_params}")

best_cat_params = {
    **study_cat.best_params,
    'iterations': 11000,
    'bootstrap_type': 'Bayesian',
    'loss_function': 'Logloss',
    'eval_metric': 'Logloss',
    'auto_class_weights': 'Balanced',
    'task_type': 'GPU',
    'gpu_ram_part': 0.85,
    'random_seed': SEED,
    'allow_writing_files': False,
    'verbose': 0,
}

oof_cat_tuned, tp_cat_tuned, auc_cat_tuned, t_cat = train_catboost_model(
    best_cat_params, X_cat_df, y, X_test_cat_df, 'CAT_tuned',
    CAT_INDICES, splits=splits_5fold
)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"\n[CAT_tuned] OOF AUC = {auc_cat_tuned:.6f}")
results['CAT_tuned'] = (oof_cat_tuned, tp_cat_tuned, auc_cat_tuned, t_cat)
joblib.dump(study_cat.best_params, './outputs/cat_best_params.pkl')

Best trial: 28. Best value: 0.962336:  80%|████████  | 32/40 [3:01:43<45:25, 340.72s/it, 10903.01/10800 seconds]  



CatBoost best trial AUC: 0.962336 (10903s)
Best params: {'learning_rate': 0.02773613701253738, 'depth': 10, 'l2_leaf_reg': 13.298930201750029, 'random_strength': 0.6417816378715336, 'bagging_temperature': 0.5683866578535111}
[CAT_tuned] OOF AUC = 0.964442 | Time = 1682.4s

[CAT_tuned] OOF AUC = 0.964442


['./outputs/cat_best_params.pkl']

In [11]:
# ── Print full comparison table ──
print("\n" + "="*70)
print("ALL MODELS — BASELINE vs TUNED")
print("="*70)
comp_df = pd.DataFrame({
    'Model': list(results.keys()),
    'OOF_AUC': [v[2] for v in results.values()],
    'Time_s': [v[3] for v in results.values()],
}).sort_values('OOF_AUC', ascending=False).reset_index(drop=True)
comp_df.index = comp_df.index + 1
comp_df.index.name = 'Rank'
print(comp_df.to_string())


ALL MODELS — BASELINE vs TUNED
             Model   OOF_AUC       Time_s
Rank                                     
1        CAT_tuned  0.964442  1682.372122
2     CAT_baseline  0.964358  2125.643430
3        XGB_tuned  0.963812   460.460407
4        LGB_tuned  0.963585   928.850726
5     LGB_baseline  0.962602   423.364995
6     XGB_baseline  0.961654   169.849523
7     MLP_baseline  0.953092  1023.019094
8      RF_baseline  0.947778   641.030293
9      LR_baseline  0.912567    45.836050


## Part 4: Ensemble

4.1 Filter models by AUC threshold (98% of max AUC).   
4.2 Weighted average (softmax of AUC).   
4.3 Stacking with Logistic Regression meta-learner.   
4.4 Pick the best.

### 4.1 Model Selection & Ensemble

In [12]:
all_aucs = [(name, v[2]) for name, v in results.items()]
all_aucs.sort(key=lambda x: -x[1])
print("All models by AUC:")
for n, a in all_aucs:
    print(f"  {n}: {a:.6f}")

selected = [name for name, _ in all_aucs[:3]]
print(f"\nSelected top-3: {selected}")
for n in selected:
    print(f"  {n}: AUC = {results[n][2]:.6f}")

All models by AUC:
  CAT_tuned: 0.964442
  CAT_baseline: 0.964358
  XGB_tuned: 0.963812
  LGB_tuned: 0.963585
  LGB_baseline: 0.962602
  XGB_baseline: 0.961654
  MLP_baseline: 0.953092
  RF_baseline: 0.947778
  LR_baseline: 0.912567

Selected top-3: ['CAT_tuned', 'CAT_baseline', 'XGB_tuned']
  CAT_tuned: AUC = 0.964442
  CAT_baseline: AUC = 0.964358
  XGB_tuned: AUC = 0.963812


In [13]:
# ── 4.2 Weighted Average (softmax of AUC) ──
aucs_sel = np.array([results[n][2] for n in selected])
# Softmax with temperature to sharpen
weights = np.exp(aucs_sel * 100) / np.exp(aucs_sel * 100).sum()
print("Ensemble weights:")
for n, w in zip(selected, weights):
    print(f"  {n}: {w:.4f}")

oof_wa = sum(results[n][0] * w for n, w in zip(selected, weights))
test_wa = sum(results[n][1] * w for n, w in zip(selected, weights))
auc_wa = roc_auc_score(y, oof_wa)
print(f"\nWeighted Average OOF AUC: {auc_wa:.6f}")

Ensemble weights:
  CAT_tuned: 0.3412
  CAT_baseline: 0.3384
  XGB_tuned: 0.3204

Weighted Average OOF AUC: 0.965111


In [14]:
oof_stack_X = np.column_stack([results[n][0] for n in selected])
test_stack_X = np.column_stack([results[n][1] for n in selected])

splits_meta = get_splits(oof_stack_X, y, groups=groups, n_folds=N_FOLDS, seed=SEED)

# (a) LR meta
oof_stack_lr = np.zeros(len(y))
test_stack_lr = np.zeros(test_stack_X.shape[0])
for trn_idx, val_idx in splits_meta:
    meta = LogisticRegression(max_iter=1000, C=1.0)
    meta.fit(oof_stack_X[trn_idx], y[trn_idx])
    oof_stack_lr[val_idx] = meta.predict_proba(oof_stack_X[val_idx])[:, 1]
    test_stack_lr += meta.predict_proba(test_stack_X)[:, 1] / N_FOLDS
auc_stack_lr = roc_auc_score(y, oof_stack_lr)
print(f"Stacking (LR meta) OOF AUC: {auc_stack_lr:.6f}")

# (b) LGB meta — shallow + regularized
oof_stack_lgb = np.zeros(len(y))
test_stack_lgb = np.zeros(test_stack_X.shape[0])
lgb_meta_params = {
    'objective': 'binary', 'metric': 'auc', 'verbosity': -1,
    'learning_rate': 0.03, 'num_leaves': 7, 'max_depth': 3,
    'min_child_samples': 100, 'reg_lambda': 1.0,
    'n_jobs': -1, 'random_state': SEED,
}
for trn_idx, val_idx in splits_meta:
    dtr = lgb.Dataset(oof_stack_X[trn_idx], y[trn_idx])
    dvl = lgb.Dataset(oof_stack_X[val_idx], y[val_idx], reference=dtr)
    m = lgb.train(lgb_meta_params, dtr, num_boost_round=500,
                  valid_sets=[dvl],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_stack_lgb[val_idx] = m.predict(oof_stack_X[val_idx])
    test_stack_lgb += m.predict(test_stack_X) / N_FOLDS
auc_stack_lgb = roc_auc_score(y, oof_stack_lgb)
print(f"Stacking (LGB meta) OOF AUC: {auc_stack_lgb:.6f}")

if auc_stack_lgb > auc_stack_lr:
    oof_stack_pred, test_stack_pred, auc_stack = oof_stack_lgb, test_stack_lgb, auc_stack_lgb
    stack_meta_name = 'LGB'
else:
    oof_stack_pred, test_stack_pred, auc_stack = oof_stack_lr, test_stack_lr, auc_stack_lr
    stack_meta_name = 'LR'
print(f"\n>>> Best meta-learner: {stack_meta_name} (AUC = {auc_stack:.6f})")

Stacking (LR meta) OOF AUC: 0.965032
Stacking (LGB meta) OOF AUC: 0.965175

>>> Best meta-learner: LGB (AUC = 0.965175)


In [15]:
from scipy.stats import rankdata

# Rank average — normalized to [0, 1]
oof_rank_raw  = np.mean([rankdata(results[n][0]) for n in selected], axis=0)
test_rank_raw = np.mean([rankdata(results[n][1]) for n in selected], axis=0)
oof_rank  = (oof_rank_raw  - oof_rank_raw.min())  / (oof_rank_raw.max()  - oof_rank_raw.min())
test_rank = (test_rank_raw - test_rank_raw.min()) / (test_rank_raw.max() - test_rank_raw.min())
auc_rank = roc_auc_score(y, oof_rank)
print(f"Rank Average OOF AUC: {auc_rank:.6f}")

# Hybrid: 50/50 rank avg + stacking
oof_hybrid  = 0.5 * oof_rank  + 0.5 * oof_stack_pred
test_hybrid = 0.5 * test_rank + 0.5 * test_stack_pred
auc_hybrid = roc_auc_score(y, oof_hybrid)
print(f"Hybrid (Rank+Stack) OOF AUC: {auc_hybrid:.6f}")

candidates = {
    'WeightedAvg':  (oof_wa,         test_wa,         auc_wa),
    'Stacking':     (oof_stack_pred, test_stack_pred, auc_stack),
    'RankAverage':  (oof_rank,       test_rank,       auc_rank),
    'Hybrid':       (oof_hybrid,     test_hybrid,     auc_hybrid),
}

print("\n" + "="*60)
print("ENSEMBLE METHOD COMPARISON")
print("="*60)
for name, (_, _, auc_) in sorted(candidates.items(), key=lambda x: -x[1][2]):
    print(f"  {name:15s}  OOF AUC = {auc_:.6f}")

for name, (oof_, tp_, auc_) in candidates.items():
    np.save(f'./outputs/test_{name}.npy', tp_)
    np.save(f'./outputs/oof_{name}.npy', oof_)

sorted_cands = sorted(candidates.items(), key=lambda x: -x[1][2])
top1_name, (_, top1_test, top1_auc) = sorted_cands[0]
top2_name, (_, top2_test, top2_auc) = sorted_cands[1]

final_test_preds = top1_test
final_method = top1_name
final_auc = top1_auc

np.save('./outputs/test_final.npy', final_test_preds)
np.save('./outputs/test_final_alt.npy', top2_test)

print(f"\n>>> Primary submission:   {top1_name} (AUC = {top1_auc:.6f})")
print(f">>> Alternate submission: {top2_name} (AUC = {top2_auc:.6f})")

Rank Average OOF AUC: 0.965087
Hybrid (Rank+Stack) OOF AUC: 0.965217

ENSEMBLE METHOD COMPARISON
  Hybrid           OOF AUC = 0.965217
  Stacking         OOF AUC = 0.965175
  WeightedAvg      OOF AUC = 0.965111
  RankAverage      OOF AUC = 0.965087

>>> Primary submission:   Hybrid (AUC = 0.965217)
>>> Alternate submission: Stacking (AUC = 0.965175)


## Part 5: Save Artifacts

In [16]:
for name in selected:
    oof, tp, auc, t = results[name][:4]
    np.save(f'./outputs/oof_{name}.npy', oof)
    np.save(f'./outputs/test_{name}.npy', tp)

joblib.dump({
    'selected': selected,
    'weights': weights.tolist(),
    'primary_method': top1_name,
    'primary_auc': top1_auc,
    'alternate_method': top2_name,
    'alternate_auc': top2_auc,
    'stack_meta': stack_meta_name,
    'all_candidates_auc': {n: c[2] for n, c in candidates.items()},
}, './outputs/ensemble_config.pkl')

joblib.dump(study_lgb.best_params, './outputs/lgb_best_params.pkl')
joblib.dump(study_xgb.best_params, './outputs/xgb_best_params.pkl')
joblib.dump(study_cat.best_params, './outputs/cat_best_params.pkl')

print("Cleaned and saved.")
print(os.listdir('./outputs/'))

Cleaned and saved.
['all_features.pkl', 'baseline_results.pkl', 'cat_best_params.pkl', 'cat_cols.pkl', 'cat_indices.pkl', 'ensemble_config.pkl', 'features_list.pkl', 'groups.npy', 'lgb_best_params.pkl', 'oof_CAT_baseline.npy', 'oof_CAT_tuned.npy', 'oof_Hybrid.npy', 'oof_RankAverage.npy', 'oof_Stacking.npy', 'oof_WeightedAvg.npy', 'oof_XGB_tuned.npy', 'scaler.pkl', 'scale_pos.pkl', 'scale_pos_winners.pkl', 'test_CAT_baseline.npy', 'test_CAT_tuned.npy', 'test_fe.csv', 'test_final.npy', 'test_final_alt.npy', 'test_Hybrid.npy', 'test_ids.npy', 'test_RankAverage.npy', 'test_Stacking.npy', 'test_WeightedAvg.npy', 'test_XGB_tuned.npy', 'train_fe.csv', 'X.npy', 'xgb_best_params.pkl', 'X_cat_df.parquet', 'X_filled.npy', 'X_scaled.npy', 'X_test.npy', 'X_test_cat_df.parquet', 'X_test_filled.npy', 'X_test_scaled.npy', 'y.npy']


### Final Summary

In [17]:
# ── Final comparison table ──
print("="*75)
print("FINAL MODEL RANKING")
print("="*75)
all_results = dict(results)
all_results['Ensemble_WA']       = (oof_wa,         test_wa,         auc_wa,     0)
all_results['Ensemble_Stack']    = (oof_stack_pred, test_stack_pred, auc_stack,  0)
all_results['Ensemble_Rank']     = (oof_rank,       test_rank,       auc_rank,   0)
all_results['Ensemble_Hybrid']   = (oof_hybrid,     test_hybrid,     auc_hybrid, 0)

final_df = pd.DataFrame({
    'Model': list(all_results.keys()),
    'OOF_AUC': [v[2] for v in all_results.values()],
}).sort_values('OOF_AUC', ascending=False).reset_index(drop=True)
final_df.index = final_df.index + 1
final_df.index.name = 'Rank'
print(final_df.to_string())
print(f"\n>>> Primary:   {final_method} | AUC = {final_auc:.6f}")
print(f">>> Alternate: {top2_name} | AUC = {top2_auc:.6f}")

FINAL MODEL RANKING
                Model   OOF_AUC
Rank                           
1     Ensemble_Hybrid  0.965217
2      Ensemble_Stack  0.965175
3         Ensemble_WA  0.965111
4       Ensemble_Rank  0.965087
5           CAT_tuned  0.964442
6        CAT_baseline  0.964358
7           XGB_tuned  0.963812
8           LGB_tuned  0.963585
9        LGB_baseline  0.962602
10       XGB_baseline  0.961654
11       MLP_baseline  0.953092
12        RF_baseline  0.947778
13        LR_baseline  0.912567

>>> Primary:   Hybrid | AUC = 0.965217
>>> Alternate: Stacking | AUC = 0.965175
